- **Гипотеза 2.** Повлияет ли уменьшение количества станций на прибыль аренды велосипедов, если убрать не ликвидные станции.

- сначала найти не ликвидные станции из файла daily_rent_detail
- затем сравнить продожительность поездки (по уровню) со средними станциями
- посмотреть сколько станции работали и сколько они принесли (в минутах)
- критерии как долго станция работала
- станция открыта более 3 месяцев

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

- читаю файл data_after_preprocessing/daily_rent_detail.parquet

In [2]:
daily_rent_detail = pd.read_parquet('../../../data_after_preprocessing/daily_rent_detail.parquet')
daily_rent_detail

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086667,117C27DBDD138B72,electric_bike,2024-08-01 08:10:22.293,2024-08-01 08:10:37.641,NaN,0,NaN,0,38.889999,-77.000000,38.889999,-77.000000,member,0.255800
16086668,4774F4D630258482,electric_bike,2024-08-08 10:05:21.938,2024-08-08 10:21:46.654,NaN,0,NaN,0,38.900002,-77.000000,38.869999,-76.949997,member,16.411934
16086669,D75836E25E77B5EC,electric_bike,2024-08-03 16:29:32.252,2024-08-03 16:35:43.179,NaN,0,NaN,0,38.919998,-76.989998,38.919998,-77.000000,member,6.182117
16086670,3B888603D18116DC,electric_bike,2024-08-03 02:49:45.661,2024-08-03 02:59:56.877,NaN,0,NaN,0,38.919998,-77.019997,38.919998,-77.029999,member,10.186934


- для начала нужно убрать станцию которая равна 0

In [3]:
daily_rent_detail = daily_rent_detail[(daily_rent_detail['start_station_id'] != 0) & (daily_rent_detail['end_station_id'] != 0)]
daily_rent_detail

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16083915,0B1E598A0BB5267A,classic_bike,2024-08-22 18:12:12.444,2024-08-22 18:59:58.532,Braddock Rd Metro North,31047,Braddock Rd Metro North,31047,38.814575,-77.052811,38.814575,-77.052811,casual,47.768135
16084580,F5F912CEC7167D79,classic_bike,2024-08-23 11:58:30.749,2024-08-23 12:00:33.749,10th & H St NE,31672,10th & H St NE,31672,38.899982,-76.992722,38.899982,-76.992722,casual,2.050000
16085242,9E2EEBF255746B1E,electric_bike,2024-08-22 13:09:17.036,2024-08-22 13:35:13.814,Wisconsin Ave & Newark St NW,31302,Wisconsin Ave & Newark St NW,31302,38.934879,-77.072754,38.934879,-77.072754,member,25.946301
16085464,2CBC6A5EB1668ABB,electric_bike,2024-08-17 09:24:19.031,2024-08-17 09:41:27.044,14th & Belmont St NW,31119,Wisconsin Ave & Newark St NW,31302,38.920807,-77.031548,38.934879,-77.072754,casual,17.133551


- добавляю столбец date

In [4]:
daily_rent_detail['date'] = daily_rent_detail['started_at'].dt.date

- группирую данные по столбцу start_station_id и считаю уникальные значения для столбца date

In [5]:
df_year_work = daily_rent_detail.groupby('start_station_id')['date'].unique()

- считаю сколько проработала станция

In [6]:
df_year_work = df_year_work.apply(lambda x: len(x))

- выбираю только те станции, которые проработали 365 дней и более

In [7]:
station_id = df_year_work[df_year_work >= 365].index

- смотрю сколько станций получилось

In [8]:
len(station_id)

687

In [9]:
station_id

Index([30200, 30201, 31000, 31001, 31002, 31003, 31004, 31005, 31006, 31007,
       ...
       32601, 32602, 32603, 32604, 32605, 32606, 32607, 32608, 32609, 33200],
      dtype='int32', name='start_station_id', length=687)

- выбираю все станции, которые проработали более 365 из DataFrame daily_rent_detail

In [10]:
station_work_more_year = daily_rent_detail[daily_rent_detail['start_station_id'].isin(station_id)]

- вывел результат

In [11]:
station_work_more_year

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes,date
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301,2020-05-30
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334,2020-05-09
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333,2020-05-24
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000,2020-05-27
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001,2020-05-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16083915,0B1E598A0BB5267A,classic_bike,2024-08-22 18:12:12.444,2024-08-22 18:59:58.532,Braddock Rd Metro North,31047,Braddock Rd Metro North,31047,38.814575,-77.052811,38.814575,-77.052811,casual,47.768135,2024-08-22
16084580,F5F912CEC7167D79,classic_bike,2024-08-23 11:58:30.749,2024-08-23 12:00:33.749,10th & H St NE,31672,10th & H St NE,31672,38.899982,-76.992722,38.899982,-76.992722,casual,2.050000,2024-08-23
16085242,9E2EEBF255746B1E,electric_bike,2024-08-22 13:09:17.036,2024-08-22 13:35:13.814,Wisconsin Ave & Newark St NW,31302,Wisconsin Ave & Newark St NW,31302,38.934879,-77.072754,38.934879,-77.072754,member,25.946301,2024-08-22
16085464,2CBC6A5EB1668ABB,electric_bike,2024-08-17 09:24:19.031,2024-08-17 09:41:27.044,14th & Belmont St NW,31119,Wisconsin Ave & Newark St NW,31302,38.920807,-77.031548,38.934879,-77.072754,casual,17.133551,2024-08-17


- создаю DataFrame

In [12]:
df = pd.DataFrame()

- написал цикл который, соберет станции, которые работали последний год

In [13]:
for station in station_work_more_year['start_station_id'].unique():
    temp_df = station_work_more_year[station_work_more_year['start_station_id'] == station]
    latest = temp_df['date'].max()
    date_year_ago = latest - pd.Timedelta(days=365)
    station_work_year = temp_df[temp_df['date'] >= date_year_ago]
    df = pd.concat([df, station_work_year], axis=0, ignore_index=False)

- смотрю сколько станций получилось

In [14]:
len(df['start_station_id'].unique())

687

- вывожу DataFrame

In [15]:
df

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes,date
10796643,2A0B26291F4BFAD4,electric_bike,2023-09-08 08:56:36.000,2023-09-08 09:41:50.000,Anacostia Library,31804,Anacostia Metro,31801,38.865768,-76.978363,38.862602,-76.994576,member,45.233334,2023-09-08
10797191,FDA2AF056320FCDB,electric_bike,2023-09-24 14:21:49.000,2023-09-24 14:33:45.000,Anacostia Library,31804,4th & East Capitol St NE,31618,38.865849,-76.978432,38.889954,-77.000351,member,11.933333,2023-09-24
10802263,57DC9B4CAB3767BC,classic_bike,2023-09-04 10:30:22.000,2023-09-04 10:33:19.000,Anacostia Library,31804,Good Hope Rd & MLK Ave SE,31802,38.865784,-76.978401,38.867374,-76.988037,member,2.950000,2023-09-04
10802471,BD9EF7AE0CB8298D,classic_bike,2023-09-06 20:46:31.000,2023-09-06 21:00:33.000,Anacostia Library,31804,1st & N St SE,31209,38.865784,-76.978401,38.875023,-77.005753,member,14.033334,2023-09-06
10808152,C9E2EABFFACA8616,electric_bike,2023-09-24 12:40:32.000,2023-09-24 12:53:15.000,Anacostia Library,31804,Eastern Market Metro / Pennsylvania Ave & 8th ...,31613,38.865768,-76.978386,38.884094,-76.995232,member,12.716666,2023-09-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16041492,42FF32E50D27D8CF,electric_bike,2024-08-13 18:35:33.179,2024-08-13 18:42:03.162,10th St & Spring Rd NW,31426,9th & Ingraham St NW,31424,38.936230,-77.027161,38.953663,-77.025612,casual,6.499717,2024-08-13
16041605,C2079F121A9028CD,electric_bike,2024-08-27 20:28:29.736,2024-08-27 20:32:54.820,10th St & Spring Rd NW,31426,Georgia Ave & Fairmont St NW,31207,38.936230,-77.027161,38.924786,-77.022331,member,4.418067,2024-08-27
16042494,CA08110A1597CFAA,electric_bike,2024-08-19 14:43:53.818,2024-08-19 14:48:17.119,10th St & Spring Rd NW,31426,Georgia Ave & Fairmont St NW,31207,38.936230,-77.027161,38.924786,-77.022331,member,4.388350,2024-08-19
16042495,1B20587C1D9C23B8,electric_bike,2024-08-26 14:58:50.932,2024-08-26 15:03:09.495,10th St & Spring Rd NW,31426,Georgia Ave & Fairmont St NW,31207,38.936230,-77.027161,38.924786,-77.022331,member,4.309383,2024-08-26


- делаю группировку по start_station_id и считаю сумму по столбцу duration_in_minutes 

In [16]:
df_for_plot = df.groupby('start_station_id')['duration_in_minutes'].sum().reset_index()

- вывожу результат

In [17]:
df_for_plot

,start_station_id,duration_in_minutes
0,30200,267596.000000
1,30201,236389.375000
2,31000,94943.546875
3,31001,118362.953125
4,31002,110934.757812
...,...,...
682,32606,12231.861328
683,32607,11933.730469
684,32608,14195.488281
685,32609,10136.699219


- делаю группировку по столбцу start_station_id и считаю количество дней отработала станция (сразу записываю в словарь)

In [18]:
station_work_day = df.groupby('start_station_id')['date'].nunique().to_dict()

- добавляю столбец station_work_day в DataFrame df_for_plot

In [19]:
df_for_plot['station_work_day'] = df_for_plot['start_station_id'].map(station_work_day)

- считаю сколько станция приносит за день

In [20]:
df_for_plot['mean_per_day'] = df_for_plot['duration_in_minutes'] / df_for_plot['station_work_day']

- вывожу результат

In [21]:
df_for_plot

,start_station_id,duration_in_minutes,station_work_day,mean_per_day
0,30200,267596.000000,365,733.139726
1,30201,236389.375000,366,645.872609
2,31000,94943.546875,365,260.119307
3,31001,118362.953125,334,354.380099
4,31002,110934.757812,365,303.930843
...,...,...,...,...
682,32606,12231.861328,318,38.464973
683,32607,11933.730469,262,45.548590
684,32608,14195.488281,291,48.781747
685,32609,10136.699219,289,35.075084


In [22]:
result = df_for_plot.sort_values(by=['mean_per_day', 'station_work_day'], ascending=True).head(50)

In [23]:
result = result[result['station_work_day'] > 90]

In [24]:
result

,start_station_id,duration_in_minutes,station_work_day,mean_per_day
538,32009,2008.987427,133,15.105169
637,32233,2416.548584,137,17.639041
459,31825,2097.320801,114,18.397551
645,32246,3873.353516,204,18.987027
628,32221,2814.635254,131,21.485765
615,32207,3060.768555,141,21.707578
451,31813,3098.948242,142,21.823579
457,31822,2572.479248,117,21.987002
617,32210,4130.877930,187,22.090256
613,32205,4183.043457,188,22.250231


### Предварительный вывод
- станции **32009, 32233, 31825, 32246** , рекомендуется убрать, так как они в день не набирают даже 20 минут аренды

- Что касается других станций, я хочу посмотреть на непрервность работы станции, так как не понятно как работали станции
- Я собираюсь добавить столбец где будет иформация о том сколько станция работала не прерывно

In [25]:
df

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes,date
10796643,2A0B26291F4BFAD4,electric_bike,2023-09-08 08:56:36.000,2023-09-08 09:41:50.000,Anacostia Library,31804,Anacostia Metro,31801,38.865768,-76.978363,38.862602,-76.994576,member,45.233334,2023-09-08
10797191,FDA2AF056320FCDB,electric_bike,2023-09-24 14:21:49.000,2023-09-24 14:33:45.000,Anacostia Library,31804,4th & East Capitol St NE,31618,38.865849,-76.978432,38.889954,-77.000351,member,11.933333,2023-09-24
10802263,57DC9B4CAB3767BC,classic_bike,2023-09-04 10:30:22.000,2023-09-04 10:33:19.000,Anacostia Library,31804,Good Hope Rd & MLK Ave SE,31802,38.865784,-76.978401,38.867374,-76.988037,member,2.950000,2023-09-04
10802471,BD9EF7AE0CB8298D,classic_bike,2023-09-06 20:46:31.000,2023-09-06 21:00:33.000,Anacostia Library,31804,1st & N St SE,31209,38.865784,-76.978401,38.875023,-77.005753,member,14.033334,2023-09-06
10808152,C9E2EABFFACA8616,electric_bike,2023-09-24 12:40:32.000,2023-09-24 12:53:15.000,Anacostia Library,31804,Eastern Market Metro / Pennsylvania Ave & 8th ...,31613,38.865768,-76.978386,38.884094,-76.995232,member,12.716666,2023-09-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16041492,42FF32E50D27D8CF,electric_bike,2024-08-13 18:35:33.179,2024-08-13 18:42:03.162,10th St & Spring Rd NW,31426,9th & Ingraham St NW,31424,38.936230,-77.027161,38.953663,-77.025612,casual,6.499717,2024-08-13
16041605,C2079F121A9028CD,electric_bike,2024-08-27 20:28:29.736,2024-08-27 20:32:54.820,10th St & Spring Rd NW,31426,Georgia Ave & Fairmont St NW,31207,38.936230,-77.027161,38.924786,-77.022331,member,4.418067,2024-08-27
16042494,CA08110A1597CFAA,electric_bike,2024-08-19 14:43:53.818,2024-08-19 14:48:17.119,10th St & Spring Rd NW,31426,Georgia Ave & Fairmont St NW,31207,38.936230,-77.027161,38.924786,-77.022331,member,4.388350,2024-08-19
16042495,1B20587C1D9C23B8,electric_bike,2024-08-26 14:58:50.932,2024-08-26 15:03:09.495,10th St & Spring Rd NW,31426,Georgia Ave & Fairmont St NW,31207,38.936230,-77.027161,38.924786,-77.022331,member,4.309383,2024-08-26


- делаю сортировку по start_station_id и date

In [26]:
df = df.sort_values(['start_station_id', 'date'])

- меняю тип данных столбца date

In [27]:
df['date'] = pd.to_datetime(df['date'])

- создаю переменную expected_freq разница 1 день

In [28]:
expected_freq = pd.Timedelta(days=1)

- создаю переменную max_work_days_stations - максимальное количество рабочих дней
- создаю переменную max_gaps_days_stations - максимальное количество дней простоя

In [29]:
max_work_days_stations = {}
max_gaps_days_stations = {}

- написал скрипт где считаю максимальное количество дней для каждой станции

In [30]:
for station_id in df['start_station_id'].unique():
    temp_df = pd.Series(df[df['start_station_id'] == station_id]['date'].unique()).diff().reset_index()
    temp_df['is_gap'] = temp_df[0] > expected_freq
    max_work_days = temp_df['is_gap'].cumsum().value_counts().max()

    max_work_days_stations[int(station_id)] = int(max_work_days)

In [32]:
max_work_days_stations

{30200: 265,
 30201: 366,
 31000: 248,
 31001: 97,
 31002: 235,
 31003: 222,
 31004: 127,
 31005: 258,
 31006: 284,
 31007: 366,
 31009: 152,
 31010: 175,
 31011: 225,
 31012: 65,
 31013: 175,
 31014: 150,
 31015: 175,
 31016: 225,
 31017: 235,
 31018: 145,
 31019: 228,
 31020: 202,
 31021: 228,
 31022: 176,
 31023: 157,
 31024: 235,
 31025: 225,
 31026: 225,
 31027: 366,
 31028: 102,
 31029: 249,
 31030: 225,
 31031: 224,
 31032: 72,
 31033: 72,
 31034: 224,
 31035: 110,
 31036: 97,
 31037: 250,
 31038: 168,
 31039: 366,
 31040: 118,
 31041: 105,
 31042: 150,
 31043: 96,
 31044: 108,
 31045: 94,
 31046: 37,
 31047: 85,
 31048: 65,
 31049: 175,
 31050: 86,
 31051: 216,
 31052: 105,
 31053: 126,
 31054: 15,
 31055: 114,
 31056: 134,
 31057: 38,
 31058: 37,
 31059: 75,
 31060: 151,
 31061: 30,
 31062: 183,
 31063: 43,
 31064: 211,
 31065: 223,
 31066: 44,
 31067: 83,
 31068: 182,
 31069: 118,
 31070: 81,
 31071: 224,
 31072: 10,
 31073: 69,
 31074: 112,
 31075: 81,
 31076: 55,
 31077: 22

In [33]:
for station_id in df['start_station_id'].unique():
    dates = pd.Series(sorted(df.loc[df['start_station_id'] == station_id, 'date'].unique()))
    diff = dates.diff()
    
    # длительность каждого простоя в днях (разница минус обычный шаг в 1 день)
    gaps = (diff[diff > expected_freq] - expected_freq).dt.days
    gap = int(gaps.max()) if len(gaps) else 0
    max_gaps_days_stations[int(station_id)] = int(gap)

In [34]:
max_gaps_days_stations

{30200: 1,
 30201: 0,
 31000: 1,
 31001: 4,
 31002: 1,
 31003: 1,
 31004: 1,
 31005: 1,
 31006: 1,
 31007: 0,
 31009: 3,
 31010: 3,
 31011: 1,
 31012: 3,
 31013: 3,
 31014: 2,
 31015: 4,
 31016: 1,
 31017: 1,
 31018: 3,
 31019: 1,
 31020: 2,
 31021: 1,
 31022: 2,
 31023: 2,
 31024: 1,
 31025: 3,
 31026: 1,
 31027: 0,
 31028: 2,
 31029: 1,
 31030: 1,
 31031: 2,
 31032: 5,
 31033: 1,
 31034: 2,
 31035: 2,
 31036: 3,
 31037: 1,
 31038: 2,
 31039: 0,
 31040: 1,
 31041: 3,
 31042: 1,
 31043: 2,
 31044: 6,
 31045: 2,
 31046: 4,
 31047: 3,
 31048: 3,
 31049: 2,
 31050: 4,
 31051: 3,
 31052: 2,
 31053: 1,
 31054: 7,
 31055: 5,
 31056: 3,
 31057: 6,
 31058: 2,
 31059: 2,
 31060: 3,
 31061: 11,
 31062: 31,
 31063: 3,
 31064: 3,
 31065: 4,
 31066: 5,
 31067: 3,
 31068: 1,
 31069: 3,
 31070: 2,
 31071: 1,
 31072: 18,
 31073: 4,
 31074: 3,
 31075: 4,
 31076: 14,
 31077: 1,
 31078: 4,
 31079: 2,
 31080: 1,
 31081: 3,
 31082: 2,
 31083: 10,
 31084: 4,
 31085: 5,
 31086: 2,
 31087: 2,
 31088: 6,
 3108

- добавляю новый столбец max_work_days

In [36]:
df_for_plot['max_work_days'] = df_for_plot['start_station_id'].map(max_work_days_stations)

- добавляю новый столбец max_gaps_days_stations

In [37]:
df_for_plot['max_gaps_days'] = df_for_plot['start_station_id'].map(max_gaps_days_stations)

- вывожу результат

In [38]:
df_for_plot.sort_values(by=['mean_per_day'], ascending=True).head(30)

,start_station_id,duration_in_minutes,station_work_day,mean_per_day,max_work_days,max_gaps_days
538,32009,2008.987427,133,15.105169,8,23
637,32233,2416.548584,137,17.639041,6,12
459,31825,2097.320801,114,18.397551,9,33
645,32246,3873.353516,204,18.987027,23,14
269,31329,1464.130859,72,20.335151,5,107
628,32221,2814.635254,131,21.485765,5,45
615,32207,3060.768555,141,21.707578,7,26
451,31813,3098.948242,142,21.823579,5,13
457,31822,2572.479248,117,21.987002,5,33
617,32210,4130.877930,187,22.090256,13,9


### Вывод по гипотеза 2. - повлияет ли уменьшение количества станций на прибыль аренды велосипедов, если убрать не ликвидные станции.

- Я добавил столбец max_gaps_days, который показывает максимальное количество дней простоя станции.
- Стоит отметить что я не знаю по каким причина были эти дни простоя
- Эти данные лучше передать на рассмотрение руководству или тому человеку, который отвечает за открытие и дальнейшую эксплуатацию станции, который как раз и расскажет в чем причина.

- Я бы из бавился от станций с наименьшим средним временем за день, а также с максимальным расхождением между max_work_days и max_gaps_days
- Хочу отметить что max_work_days должен быть как можно больше
- А max_gaps_days должен быть как можно меньше